# 05 — Model Development

## Objetivo

Desenvolver e avaliar modelos de previsão de vendas utilizando os conjuntos de treino e validação preparados no Notebook 04.

O objetivo desta etapa é estabelecer um baseline de desempenho e, em seguida, avaliar modelos de maior capacidade preditiva, mantendo a mesma separação temporal e as mesmas variáveis de entrada para permitir uma comparação consistente.

Os modelos serão avaliados utilizando métricas de erro adequadas ao problema de previsão de vendas.



In [2]:

# ============================================================
# IMPORTS
# ============================================================

import pandas as pd
import numpy as np

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

import lightgbm as lgb

print("Bibliotecas carregadas com sucesso.")

Bibliotecas carregadas com sucesso.


In [3]:
# ============================================================
# CARREGAMENTO DOS DADOS PREPARADOS
# ============================================================

train_path = "../data/processed/train_modeling.parquet"
validation_path = "../data/processed/validation_modeling.parquet"

train_modeling = pd.read_parquet(train_path)
validation_modeling = pd.read_parquet(validation_path)

print("Dados carregados com sucesso.")

print(f"Treino: {train_modeling.shape}")
print(f"Validação: {validation_modeling.shape}")

Dados carregados com sucesso.
Treino: (5661993, 26)
Validação: (85372, 26)


In [4]:
# ============================================================
# SEPARAÇÃO ENTRE FEATURES E VARIÁVEL ALVO
# ============================================================

target = "sales"

X_train = train_modeling.drop(columns=[target])
y_train = train_modeling[target]

X_validation = validation_modeling.drop(columns=[target])
y_validation = validation_modeling[target]

print("Dimensões:")
print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")
print(f"X_validation: {X_validation.shape}")
print(f"y_validation: {y_validation.shape}")

print("\nVariável alvo:")
print(target)

print("\nQuantidade de features:")
print(X_train.shape[1])

Dimensões:
X_train: (5661993, 25)
y_train: (5661993,)
X_validation: (85372, 25)
y_validation: (85372,)

Variável alvo:
sales

Quantidade de features:
25


## Estratégia de Modelagem

A modelagem será conduzida de forma incremental, iniciando com um baseline simples para estabelecer uma referência de desempenho.

Em seguida, serão avaliados modelos capazes de capturar relações não lineares entre as variáveis históricas, temporais, categóricas e de calendário.

A comparação será realizada utilizando o mesmo conjunto de validação temporal, evitando alterações na divisão dos dados durante a avaliação.

O desempenho dos modelos será analisado por meio de métricas de erro, permitindo identificar a abordagem mais adequada para a previsão de vendas.

In [5]:
# ============================================================
# DIAGNÓSTICO DAS FEATURES PARA MODELAGEM
# ============================================================

print("Tipos das features:")
print(X_train.dtypes)

print("\nFeatures categóricas:")
print(
    X_train.select_dtypes(
        include=["category", "object", "str"]
    ).columns.tolist()
)

print("\nFeatures numéricas:")
print(
    X_train.select_dtypes(
        include=["number"]
    ).columns.tolist()
)

print("\nQuantidade total de features:")
print(X_train.shape[1])

Tipos das features:
item_id            category
store_id           category
d                       str
lag_1               float32
lag_7               float32
lag_14              float32
lag_28              float32
wday                   int8
weekday            category
month                  int8
year                  int16
week_of_year           int8
snap_CA                int8
snap_TX                int8
snap_WI                int8
snap                   int8
event_name_1       category
event_type_1       category
event_name_2       category
event_type_2       category
day_num               int16
_day_num              int32
rolling_mean_7      float32
rolling_mean_14     float32
rolling_mean_28     float32
dtype: object

Features categóricas:
['item_id', 'store_id', 'd', 'weekday', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2']

Features numéricas:
['lag_1', 'lag_7', 'lag_14', 'lag_28', 'wday', 'month', 'year', 'week_of_year', 'snap_CA', 'snap_TX', 'snap_WI', 'snap

In [6]:
# ============================================================
# BASELINE 1 — ÚLTIMO VALOR OBSERVADO
# ============================================================

y_pred_baseline_lag1 = X_validation["lag_1"].copy()

print("Baseline baseado em lag_1 criado com sucesso.")

print("\nDimensão das previsões:")
print(y_pred_baseline_lag1.shape)

print("\nPrimeiras previsões:")
print(y_pred_baseline_lag1.head())

Baseline baseado em lag_1 criado com sucesso.

Dimensão das previsões:
(85372,)

Primeiras previsões:
0    0.0
1    2.0
2    1.0
3    1.0
4    0.0
Name: lag_1, dtype: float32


In [7]:
# ============================================================
# AVALIAÇÃO DO BASELINE 1
# ============================================================

mae_baseline_lag1 = mean_absolute_error(
    y_validation,
    y_pred_baseline_lag1
)

rmse_baseline_lag1 = np.sqrt(
    mean_squared_error(
        y_validation,
        y_pred_baseline_lag1
    )
)

print("Resultados do Baseline 1 — lag_1")
print(f"MAE:  {mae_baseline_lag1:.4f}")
print(f"RMSE: {rmse_baseline_lag1:.4f}")

Resultados do Baseline 1 — lag_1
MAE:  1.2817
RMSE: 2.7682


## Baseline 2 — Sazonalidade Semanal

Como segundo baseline, será utilizada a variável `lag_7`, que representa o valor de vendas observado sete dias antes.

Essa abordagem busca capturar a sazonalidade semanal do comportamento das vendas, utilizando como previsão a venda do mesmo dia da semana anterior.

O desempenho será avaliado utilizando o mesmo conjunto de validação temporal e as mesmas métricas utilizadas no Baseline 1, permitindo uma comparação consistente entre as abordagens.

In [8]:
# ============================================================
# BASELINE 2 — LAG_7
# ============================================================

# Utiliza a venda de 7 dias atrás como previsão
baseline_2_pred = X_validation["lag_7"].copy()

print("Baseline baseado em lag_7 criado com sucesso.")

print("\nDimensão das previsões:")
print(baseline_2_pred.shape)

print("\nPrimeiras previsões:")
print(baseline_2_pred.head())

Baseline baseado em lag_7 criado com sucesso.

Dimensão das previsões:
(85372,)

Primeiras previsões:
0    0.0
1    2.0
2    2.0
3    0.0
4    1.0
Name: lag_7, dtype: float32


In [9]:
# ============================================================
# AVALIAÇÃO DO BASELINE 2
# ============================================================

mae_baseline_2 = mean_absolute_error(
    y_validation,
    baseline_2_pred
)

mse_baseline_2 = mean_squared_error(
    y_validation,
    baseline_2_pred
)

rmse_baseline_2 = np.sqrt(mse_baseline_2)

print("Resultados do Baseline 2 — lag_7")
print(f"MAE:  {mae_baseline_2:.4f}")
print(f"RMSE: {rmse_baseline_2:.4f}")

Resultados do Baseline 2 — lag_7
MAE:  1.3097
RMSE: 2.8066


In [10]:
# ============================================================
# COMPARAÇÃO DOS BASELINES
# ============================================================

baseline_results = pd.DataFrame({
    "Modelo": [
        "Baseline — lag_1",
        "Baseline — lag_7"
    ],
    "MAE": [
        mae_baseline_lag1,
        mae_baseline_2
    ],
    "RMSE": [
        rmse_baseline_lag1,
        rmse_baseline_2
    ]
})

display(baseline_results)

,Modelo,MAE,RMSE
0,Baseline — lag_1,1.281732,2.768171
1,Baseline — lag_7,1.309680,2.806559


In [11]:
# ============================================================
# CARREGAMENTO DOS DADOS PREPARADOS
# ============================================================

train_path = "../data/processed/train_modeling.parquet"
validation_path = "../data/processed/validation_modeling.parquet"

train_modeling = pd.read_parquet(train_path)
validation_modeling = pd.read_parquet(validation_path)

print("Dados carregados com sucesso.")
print(f"Treino: {train_modeling.shape}")
print(f"Validação: {validation_modeling.shape}")

Dados carregados com sucesso.
Treino: (5661993, 26)
Validação: (85372, 26)


In [12]:
# ============================================================
# CONFERÊNCIA DOS DADOS PARA MODELAGEM
# ============================================================

print("Colunas de treino:")
print(train_modeling.columns.tolist())

print("\nValores ausentes — treino:")
print(train_modeling.isna().sum().sum())

print("\nValores ausentes — validação:")
print(validation_modeling.isna().sum().sum())

print("\nTipos das variáveis:")
print(train_modeling.dtypes)

Colunas de treino:
['item_id', 'store_id', 'd', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'wday', 'weekday', 'month', 'year', 'week_of_year', 'snap_CA', 'snap_TX', 'snap_WI', 'snap', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'day_num', '_day_num', 'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28', 'sales']

Valores ausentes — treino:
0

Valores ausentes — validação:
0

Tipos das variáveis:
item_id            category
store_id           category
d                       str
lag_1               float32
lag_7               float32
lag_14              float32
lag_28              float32
wday                   int8
weekday            category
month                  int8
year                  int16
week_of_year           int8
snap_CA                int8
snap_TX                int8
snap_WI                int8
snap                   int8
event_name_1       category
event_type_1       category
event_name_2       category
event_type_2       category
day_num               in

In [13]:
# ============================================================
# DEFINIÇÃO DAS FEATURES PARA O MODELO
# ============================================================

target = "sales"

exclude_cols = [
    target,
    "d"
]

feature_cols = [
    col
    for col in train_modeling.columns
    if col not in exclude_cols
]

X_train = train_modeling[feature_cols].copy()
y_train = train_modeling[target].copy()

X_validation = validation_modeling[feature_cols].copy()
y_validation = validation_modeling[target].copy()

print(f"Quantidade de features: {len(feature_cols)}")
print(f"X_train: {X_train.shape}")
print(f"X_validation: {X_validation.shape}")

print("\nFeatures:")
print(feature_cols)

Quantidade de features: 24
X_train: (5661993, 24)
X_validation: (85372, 24)

Features:
['item_id', 'store_id', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'wday', 'weekday', 'month', 'year', 'week_of_year', 'snap_CA', 'snap_TX', 'snap_WI', 'snap', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'day_num', '_day_num', 'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28']


In [14]:
# ============================================================
# DEFINIÇÃO DAS VARIÁVEIS CATEGÓRICAS
# ============================================================

categorical_features = X_train.select_dtypes(
    include=["category"]
).columns.tolist()

print("Variáveis categóricas:")
print(categorical_features)

print(f"\nQuantidade: {len(categorical_features)}")

Variáveis categóricas:
['item_id', 'store_id', 'weekday', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2']

Quantidade: 7


In [15]:
# ============================================================
# VERIFICAÇÃO DAS CATEGORIAS
# ============================================================

for col in categorical_features:
    train_categories = set(X_train[col].cat.categories)
    validation_categories = set(X_validation[col].cat.categories)

    missing_in_validation = train_categories - validation_categories
    missing_in_train = validation_categories - train_categories

    print(f"\n{col}")
    print(f"Categorias no treino: {len(train_categories)}")
    print(f"Categorias na validação: {len(validation_categories)}")
    print(f"Ausentes na validação: {len(missing_in_validation)}")
    print(f"Ausentes no treino: {len(missing_in_train)}")


item_id
Categorias no treino: 3049
Categorias na validação: 3049
Ausentes na validação: 0
Ausentes no treino: 0

store_id
Categorias no treino: 10
Categorias na validação: 10
Ausentes na validação: 0
Ausentes no treino: 0

weekday
Categorias no treino: 7
Categorias na validação: 7
Ausentes na validação: 0
Ausentes no treino: 0

event_name_1
Categorias no treino: 31
Categorias na validação: 31
Ausentes na validação: 0
Ausentes no treino: 0

event_type_1
Categorias no treino: 5
Categorias na validação: 5
Ausentes na validação: 0
Ausentes no treino: 0

event_name_2
Categorias no treino: 5
Categorias na validação: 5
Ausentes na validação: 0
Ausentes no treino: 0

event_type_2
Categorias no treino: 3
Categorias na validação: 3
Ausentes na validação: 0
Ausentes no treino: 0


In [16]:
# ============================================================
# PADRONIZAÇÃO DAS VARIÁVEIS CATEGÓRICAS
# ============================================================

for col in categorical_features:
    X_validation[col] = X_validation[col].astype(
        X_train[col].dtype
    )

print("Tipos categóricos padronizados com base no treino.")

print("\nVerificação:")
for col in categorical_features:
    print(
        f"{col}: "
        f"{X_train[col].dtype} | "
        f"{X_validation[col].dtype}"
    )

Tipos categóricos padronizados com base no treino.

Verificação:
item_id: category | category
store_id: category | category
weekday: category | category
event_name_1: category | category
event_type_1: category | category
event_name_2: category | category
event_type_2: category | category


In [17]:
# ============================================================
# CONFIGURAÇÃO DO MODELO LIGHTGBM
# ============================================================

model_lgbm = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

print("Modelo LightGBM configurado.")

Modelo LightGBM configurado.


In [18]:
# ============================================================
# TREINAMENTO DO MODELO LIGHTGBM
# ============================================================

model_lgbm.fit(
    X_train,
    y_train,
    categorical_feature=categorical_features,
    eval_set=[(X_validation, y_validation)],
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            verbose=True
        )
    ]
)

print("\nTreinamento concluído.")

c:\Projetos\retail-demand-forecasting\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.267646 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4778
[LightGBM] [Info] Number of data points in the train set: 5661993, number of used features: 24
[LightGBM] [Info] Start training from score 1.320919
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[495]	valid_0's l2: 4.03875

Treinamento concluído.


In [19]:
# ============================================================
# PREVISÕES DO MODELO LIGHTGBM
# ============================================================

y_pred_lgbm = model_lgbm.predict(X_validation)

print("Previsões geradas com sucesso.")

print(f"Dimensão das previsões: {y_pred_lgbm.shape}")

print("\nPrimeiras previsões:")
print(y_pred_lgbm[:5])

Previsões geradas com sucesso.
Dimensão das previsões: (85372,)

Primeiras previsões:
[0.79338551 0.90761315 0.89690007 0.89841352 0.9079214 ]


In [20]:
# ============================================================
# AVALIAÇÃO DO MODELO LIGHTGBM
# ============================================================

mae_lgbm = mean_absolute_error(
    y_validation,
    y_pred_lgbm
)

mse_lgbm = mean_squared_error(
    y_validation,
    y_pred_lgbm
)

rmse_lgbm = np.sqrt(mse_lgbm)

print("Resultados do LightGBM")
print(f"MAE:  {mae_lgbm:.4f}")
print(f"RMSE: {rmse_lgbm:.4f}")

Resultados do LightGBM
MAE:  1.0372
RMSE: 2.0097


In [21]:
# ============================================================
# COMPARAÇÃO DOS MODELOS
# ============================================================

model_results = pd.DataFrame({
    "Modelo": [
        "Baseline — lag_1",
        "Baseline — lag_7",
        "LightGBM"
    ],
    "MAE": [
        1.281732,
        1.309680,
        1.0372
    ],
    "RMSE": [
        2.768171,
        2.806559,
        2.0097
    ]
})

display(
    model_results.sort_values("MAE")
)


,Modelo,MAE,RMSE
2,LightGBM,1.037200,2.009700
0,Baseline — lag_1,1.281732,2.768171
1,Baseline — lag_7,1.309680,2.806559


In [22]:
# ============================================================
# GANHO DO LIGHTGBM SOBRE O MELHOR BASELINE
# ============================================================

best_baseline_mae = model_results.loc[
    model_results["Modelo"] == "Baseline — lag_1",
    "MAE"
].iloc[0]

best_baseline_rmse = model_results.loc[
    model_results["Modelo"] == "Baseline — lag_1",
    "RMSE"
].iloc[0]

mae_lgbm_result = model_results.loc[
    model_results["Modelo"] == "LightGBM",
    "MAE"
].iloc[0]

rmse_lgbm_result = model_results.loc[
    model_results["Modelo"] == "LightGBM",
    "RMSE"
].iloc[0]

mae_improvement = (
    (best_baseline_mae - mae_lgbm_result)
    / best_baseline_mae
) * 100

rmse_improvement = (
    (best_baseline_rmse - rmse_lgbm_result)
    / best_baseline_rmse
) * 100

print(f"Ganho de MAE:  {mae_improvement:.2f}%")
print(f"Ganho de RMSE: {rmse_improvement:.2f}%")

Ganho de MAE:  19.08%
Ganho de RMSE: 27.40%


In [23]:
# ============================================================
# IMPORTÂNCIA DAS FEATURES
# ============================================================

feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model_lgbm.feature_importances_
})

feature_importance = feature_importance.sort_values(
    "importance",
    ascending=False
)

display(feature_importance.head(15))

,feature,importance
0,item_id,2267
2,lag_1,1789
21,rolling_mean_7,1407
5,lag_28,1260
4,lag_14,1120
3,lag_7,1114
19,day_num,1076
10,week_of_year,869
22,rolling_mean_14,821
23,rolling_mean_28,724


In [24]:
# ============================================================
# ANÁLISE DOS ERROS DE PREVISÃO
# ============================================================

error_analysis = pd.DataFrame({
    "actual": y_validation.values,
    "prediction": y_pred_lgbm
})

error_analysis["error"] = (
    error_analysis["actual"] -
    error_analysis["prediction"]
)

error_analysis["absolute_error"] = (
    error_analysis["error"].abs()
)

display(
    error_analysis.head(10)
)

print("\nErro médio:")
print(f"{error_analysis['error'].mean():.4f}")

print("\nErro absoluto médio:")
print(f"{error_analysis['absolute_error'].mean():.4f}")

print("\nMaior erro absoluto:")
print(f"{error_analysis['absolute_error'].max():.4f}")

,actual,prediction,error,absolute_error
0,2,0.793386,1.206614,1.206614
1,1,0.907613,0.092387,0.092387
2,1,0.896900,0.103100,0.103100
3,0,0.898414,-0.898414,0.898414
4,4,0.907921,3.092079,3.092079
5,0,1.748477,-1.748477,1.748477
6,0,1.258068,-1.258068,1.258068
7,4,0.918924,3.081076,3.081076
8,1,1.267278,-0.267278,0.267278
9,3,1.097878,1.902122,1.902122



Erro médio:
-0.0111

Erro absoluto médio:
1.0372

Maior erro absoluto:
71.2334


In [25]:
# ============================================================
# DISTRIBUIÇÃO DOS ERROS
# ============================================================

print("Percentis do erro absoluto:")

print(
    error_analysis["absolute_error"]
    .quantile([0.50, 0.75, 0.90, 0.95, 0.99, 1.00])
)

Percentis do erro absoluto:
0.50     0.579680
0.75     1.188238
0.90     2.328812
0.95     3.524906
0.99     7.427098
1.00    71.233388
Name: absolute_error, dtype: float64


In [26]:
# ============================================================
# REGISTRO DO MODELO DE REFERÊNCIA
# ============================================================

reference_results = {
    "modelo": "LightGBM - baseline",
    "mae": mae_lgbm_result,
    "rmse": rmse_lgbm_result,
    "best_iteration": model_lgbm.best_iteration_
}

print("Modelo de referência registrado.")
print(reference_results)

Modelo de referência registrado.
{'modelo': 'LightGBM - baseline', 'mae': np.float64(1.0372), 'rmse': np.float64(2.0097), 'best_iteration': 495}


In [27]:
# ============================================================
# LIGHTGBM — MODELO AJUSTADO
# ============================================================

model_lgbm_tuned = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=100,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

print("Modelo LightGBM ajustado configurado.")

Modelo LightGBM ajustado configurado.


In [28]:
# ============================================================
# TREINAMENTO DO LIGHTGBM AJUSTADO
# ============================================================

model_lgbm_tuned.fit(
    X_train,
    y_train,
    categorical_feature=categorical_features,
    eval_set=[(X_validation, y_validation)],
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            verbose=True
        )
    ]
)

print("\nTreinamento do modelo ajustado concluído.")
print(f"Melhor iteração: {model_lgbm_tuned.best_iteration_}")

c:\Projetos\retail-demand-forecasting\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.329894 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4776
[LightGBM] [Info] Number of data points in the train set: 5661993, number of used features: 23
[LightGBM] [Info] Start training from score 1.320919
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[632]	valid_0's l2: 3.98943

Treinamento do modelo ajustado concluído.
Melhor iteração: 632


In [29]:
# ============================================================
# PREVISÕES DO LIGHTGBM AJUSTADO
# ============================================================

y_pred_lgbm_tuned = model_lgbm_tuned.predict(X_validation)

print("Previsões do modelo ajustado geradas com sucesso.")

print(f"Dimensão das previsões: {y_pred_lgbm_tuned.shape}")

print("\nPrimeiras previsões:")
print(y_pred_lgbm_tuned[:5])

Previsões do modelo ajustado geradas com sucesso.
Dimensão das previsões: (85372,)

Primeiras previsões:
[0.79490763 0.89141947 0.91262477 0.90059812 0.90787391]


In [30]:
# ============================================================
# AVALIAÇÃO DO LIGHTGBM AJUSTADO
# ============================================================

mae_lgbm_tuned = mean_absolute_error(
    y_validation,
    y_pred_lgbm_tuned
)

mse_lgbm_tuned = mean_squared_error(
    y_validation,
    y_pred_lgbm_tuned
)

rmse_lgbm_tuned = np.sqrt(mse_lgbm_tuned)

print("Resultados do LightGBM ajustado")
print(f"MAE:  {mae_lgbm_tuned:.4f}")
print(f"RMSE: {rmse_lgbm_tuned:.4f}")

Resultados do LightGBM ajustado
MAE:  1.0350
RMSE: 1.9974


In [31]:
# ============================================================
# COMPARAÇÃO DOS EXPERIMENTOS LIGHTGBM
# ============================================================

lgbm_results = pd.DataFrame({
    "Modelo": [
        "LightGBM — referência",
        "LightGBM — ajustado"
    ],
    "MAE": [
        reference_results["mae"],
        mae_lgbm_tuned
    ],
    "RMSE": [
        reference_results["rmse"],
        rmse_lgbm_tuned
    ],
    "Melhor iteração": [
        reference_results["best_iteration"],
        model_lgbm_tuned.best_iteration_
    ]
})

display(
    lgbm_results.sort_values("MAE")
)

,Modelo,MAE,RMSE,Melhor iteração
1,LightGBM — ajustado,1.035037,1.997356,632
0,LightGBM — referência,1.037200,2.009700,495


In [32]:
# ============================================================
# COMPARAÇÃO FINAL DOS MODELOS
# ============================================================

final_results = pd.DataFrame({
    "Modelo": [
        "Baseline — lag_1",
        "Baseline — lag_7",
        "LightGBM — referência",
        "LightGBM — ajustado"
    ],
    "MAE": [
        mae_baseline_lag1,
        mae_baseline_2,
        mae_lgbm,
        mae_lgbm_tuned
    ],
    "RMSE": [
        rmse_baseline_lag1,
        rmse_baseline_2,
        rmse_lgbm,
        rmse_lgbm_tuned
    ]
})

final_results = (
    final_results
    .sort_values("MAE")
    .reset_index(drop=True)
)

display(final_results)

,Modelo,MAE,RMSE
0,LightGBM — ajustado,1.035037,1.997356
1,LightGBM — referência,1.037220,2.009664
2,Baseline — lag_1,1.281732,2.768171
3,Baseline — lag_7,1.309680,2.806559


In [33]:
# ============================================================
# REGISTRO DO MODELO FINAL
# ============================================================

final_model_results = {
    "model": "LightGBM — ajustado",
    "objective": "regression",
    "n_estimators": 1000,
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_child_samples": 100,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42,
    "best_iteration": model_lgbm_tuned.best_iteration_,
    "mae": mae_lgbm_tuned,
    "rmse": rmse_lgbm_tuned
}

print("Modelo final registrado:\n")

for key, value in final_model_results.items():
    print(f"{key}: {value}")

Modelo final registrado:

model: LightGBM — ajustado
objective: regression
n_estimators: 1000
learning_rate: 0.05
num_leaves: 31
min_child_samples: 100
subsample: 0.8
colsample_bytree: 0.8
random_state: 42
best_iteration: 632
mae: 1.0350366331680052
rmse: 1.9973556630651887


## Conclusão da Modelagem

Foram avaliadas abordagens baseline baseadas em `lag_1` e `lag_7`, além de duas configurações do modelo LightGBM.

O **LightGBM ajustado** apresentou o melhor desempenho na validação temporal, com:

- **MAE:** 1.0350
- **RMSE:** 1.9974
- **Best iteration:** 632

Em comparação com o melhor baseline (`lag_1`), o modelo final apresentou ganho de:

- **19,25% no MAE**
- **27,85% no RMSE**

Dessa forma, o **LightGBM ajustado** foi selecionado como modelo de referência para as próximas etapas do projeto.

In [34]:
# ============================================================
# PERSISTÊNCIA DO MODELO FINAL
# ============================================================

from pathlib import Path
import json

models_path = Path("../models")
models_path.mkdir(exist_ok=True)

# Salvar modelo LightGBM
model_path = models_path / "lightgbm_final.txt"

model_lgbm_tuned.booster_.save_model(
    str(model_path),
    num_iteration=model_lgbm_tuned.best_iteration_
)

# Metadados do modelo
metadata = {
    "model": "LightGBM — ajustado",
    "objective": "regression",
    "best_iteration": int(model_lgbm_tuned.best_iteration_),
    "mae": float(mae_lgbm_tuned),
    "rmse": float(rmse_lgbm_tuned),
    "features": list(X_train.columns)
}

metadata_path = models_path / "lightgbm_final_metadata.json"

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=4, ensure_ascii=False)

print("Modelo final salvo com sucesso.")
print(f"Modelo: {model_path}")
print(f"Metadados: {metadata_path}")

Modelo final salvo com sucesso.
Modelo: ..\models\lightgbm_final.txt
Metadados: ..\models\lightgbm_final_metadata.json
